# Diabetes Prediction Using R

**CA-3 Machine Learning Project**

This Google Colab notebook contains the R setup, Logistic Regression model, evaluation metrics, and each visualization in a separate cell. The model predicts diabetes outcome using the Pima Indians Diabetes dataset.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq r-base
!pip install -q rpy2
%load_ext rpy2.ipython

In [ ]:
%%R

packages <- c("ggplot2", "pROC")

for (p in packages) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(ggplot2)
library(pROC)

print("R program is working successfully!")

## Main R program

Run this cell to load and preprocess the dataset, calculate the given-problem metrics, train the Logistic Regression model, evaluate it, and calculate the ROC-AUC.

In [ ]:
%%R

# =====================================================
# CA-3 MACHINE LEARNING PROJECT
# DIABETES PREDICTION USING R
# =====================================================

packages <- c("ggplot2", "pROC")
for (p in packages) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}
library(ggplot2)
library(pROC)

# PART A: GIVEN PROBLEM STATEMENT
cat("\n----- GIVEN PROBLEM STATEMENT -----\n")

TP <- 70
FN <- 10
FP <- 20
TN <- 100

accuracy <- (TP + TN) / (TP + TN + FP + FN)
precision <- TP / (TP + FP)
recall <- TP / (TP + FN)
f1_score <- 2 * precision * recall / (precision + recall)

cat("Total Patients:", TP + FN + FP + TN, "\n")
cat("True Positive:", TP, "\n")
cat("False Negative:", FN, "\n")
cat("False Positive:", FP, "\n")
cat("True Negative:", TN, "\n")
cat("\nEvaluation Metrics:\n")
cat("Accuracy :", round(accuracy * 100, 2), "%\n")
cat("Precision:", round(precision * 100, 2), "%\n")
cat("Recall   :", round(recall * 100, 2), "%\n")
cat("F1 Score :", round(f1_score * 100, 2), "%\n")

given_cm <- matrix(c(TN, FP, FN, TP), nrow = 2, byrow = TRUE)
rownames(given_cm) <- c("Actual No", "Actual Yes")
colnames(given_cm) <- c("Predicted No", "Predicted Yes")
print(given_cm)

# PART B: ACTUAL MACHINE LEARNING MODEL
cat("\n----- DIABETES PREDICTION MODEL -----\n")

url <- paste0(
  "https://raw.githubusercontent.com/",
  "jbrownlee/Datasets/master/",
  "pima-indians-diabetes.data.csv"
)
data <- read.csv(url, header = FALSE)

colnames(data) <- c(
  "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
  "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
)

cat("\nFirst six records:\n")
print(head(data))
cat("\nDataset dimensions:\n")
print(dim(data))
cat("\nDataset summary:\n")
print(summary(data))

data$OutcomeLabel <- factor(
  data$Outcome, levels = c(0, 1),
  labels = c("No Diabetes", "Diabetes")
)

# Data preprocessing: replace selected zero values with missing values,
# then replace missing values with each column's median.
cols <- c("Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI")
for (col in cols) {
  data[data[[col]] == 0, col] <- NA
}
for (col in cols) {
  data[[col]][is.na(data[[col]])] <- median(data[[col]], na.rm = TRUE)
}

data$OutcomeLabel <- NULL
data$Outcome <- factor(data$Outcome, levels = c(0, 1), labels = c("No", "Yes"))

# Stratified 80/20 training and testing split
set.seed(123)
train_index <- unlist(
  lapply(split(seq_len(nrow(data)), data$Outcome),
         function(x) sample(x, floor(0.8 * length(x))))
)
train_data <- data[train_index, ]
test_data <- data[-train_index, ]

cat("\nTraining records:", nrow(train_data), "\n")
cat("Testing records:", nrow(test_data), "\n")

# Logistic Regression: supervised binary classification
model <- glm(Outcome ~ ., data = train_data, family = binomial)
cat("\nLogistic Regression Model:\n")
print(summary(model))

probability <- predict(model, newdata = test_data, type = "response")
predicted <- factor(ifelse(probability >= 0.5, "Yes", "No"),
                    levels = c("No", "Yes"))
actual <- test_data$Outcome

model_cm <- table(Actual = actual, Predicted = predicted)
cat("\nModel Confusion Matrix:\n")
print(model_cm)

TN2 <- model_cm["No", "No"]
FP2 <- model_cm["No", "Yes"]
FN2 <- model_cm["Yes", "No"]
TP2 <- model_cm["Yes", "Yes"]

model_accuracy <- (TP2 + TN2) / sum(model_cm)
model_precision <- ifelse(TP2 + FP2 == 0, 0, TP2 / (TP2 + FP2))
model_recall <- ifelse(TP2 + FN2 == 0, 0, TP2 / (TP2 + FN2))
model_f1 <- ifelse(
  model_precision + model_recall == 0, 0,
  2 * model_precision * model_recall / (model_precision + model_recall)
)

cat("\n----- MODEL PERFORMANCE -----\n")
cat("Accuracy :", round(model_accuracy * 100, 2), "%\n")
cat("Precision:", round(model_precision * 100, 2), "%\n")
cat("Recall   :", round(model_recall * 100, 2), "%\n")
cat("F1 Score :", round(model_f1 * 100, 2), "%\n")

roc_result <- roc(actual, probability, levels = c("No", "Yes"),
                  direction = "<", quiet = TRUE)
auc_value <- as.numeric(auc(roc_result))
cat("\nROC-AUC Score:", round(auc_value, 3), "\n")

results <- data.frame(
  Actual = actual,
  Predicted = predicted,
  Probability = round(probability, 3)
)
cat("\nActual vs Predicted Results:\n")
print(head(results, 15))

cat("\n====================================\n")
cat("DIABETES PREDICTION PROJECT COMPLETED\n")
cat("====================================\n")

## Separate visualizations

Run each cell separately to display each chart individually. These cells use the objects created by the main R program above.

In [ ]:
%%R
# Install the png package as in the original Colab workflow
if (!requireNamespace("png", quietly = TRUE)) {
  install.packages("png", repos = "https://cloud.r-project.org")
}
library(ggplot2)

cm <- data.frame(
  Actual = c("No", "No", "Yes", "Yes"),
  Predicted = c("No", "Yes", "No", "Yes"),
  Count = c(100, 20, 10, 70)
)

print(
  ggplot(cm, aes(Predicted, Actual, fill = Count)) +
    geom_tile(color = "white") +
    geom_text(aes(label = Count), size = 7) +
    scale_fill_gradient(low = "lightblue", high = "blue") +
    labs(title = "Given Problem - Confusion Matrix") +
    theme_minimal()
)

In [ ]:
%%R
metrics <- data.frame(
  Metric = c("Accuracy", "Precision", "Recall", "F1 Score"),
  Score = c(85, 77.78, 87.5, 82.35)
)

print(
  ggplot(metrics, aes(Metric, Score, fill = Metric)) +
    geom_col() +
    geom_text(aes(label = paste0(Score, "%")), vjust = -0.5) +
    ylim(0, 105) +
    labs(title = "Given Problem - Performance Metrics", y = "Percentage") +
    theme_minimal()
)

In [ ]:
%%R
print(
  ggplot(data, aes(Outcome, fill = Outcome)) +
    geom_bar() +
    labs(title = "Diabetes Distribution",
         x = "Diabetes Outcome", y = "Number of Patients") +
    theme_minimal()
)

In [ ]:
%%R
print(
  ggplot(data, aes(Glucose, fill = Outcome)) +
    geom_histogram(bins = 30, alpha = 0.7, position = "identity") +
    labs(title = "Glucose Level Distribution",
         x = "Glucose Level", y = "Frequency") +
    theme_minimal()
)

In [ ]:
%%R
print(
  ggplot(data, aes(Glucose, BMI, color = Outcome)) +
    geom_point(size = 2, alpha = 0.6) +
    labs(title = "BMI vs Glucose Level", x = "Glucose", y = "BMI") +
    theme_minimal()
)

In [ ]:
%%R
print(
  ggplot(data, aes(Age, fill = Outcome)) +
    geom_histogram(bins = 25, alpha = 0.7, position = "identity") +
    labs(title = "Age Distribution", x = "Age", y = "Frequency") +
    theme_minimal()
)

In [ ]:
%%R
model_cm_data <- as.data.frame(as.table(model_cm))

print(
  ggplot(model_cm_data, aes(Predicted, Actual, fill = Freq)) +
    geom_tile(color = "white") +
    geom_text(aes(label = Freq), size = 7) +
    scale_fill_gradient(low = "lightgreen", high = "darkgreen") +
    labs(title = "Diabetes Model - Confusion Matrix") +
    theme_minimal()
)

In [ ]:
%%R
model_metrics <- data.frame(
  Metric = c("Accuracy", "Precision", "Recall", "F1 Score"),
  Score = c(model_accuracy, model_precision, model_recall, model_f1) * 100
)

print(
  ggplot(model_metrics, aes(Metric, Score, fill = Metric)) +
    geom_col() +
    geom_text(aes(label = paste0(round(Score, 2), "%")), vjust = -0.5) +
    coord_cartesian(ylim = c(0, 105)) +
    labs(title = "Diabetes Model - Performance Metrics",
         y = "Percentage") +
    theme_minimal()
)

In [ ]:
%%R
plot(roc_result, main = "ROC Curve - Diabetes Prediction",
     col = "blue", lwd = 3)
cat("AUC Score:", round(as.numeric(auc(roc_result)), 3))


In [ ]:
%%R
model_heat <- data.frame(
  Metric = c("Accuracy", "Precision", "Recall", "F1 Score"),
  Value = c(model_accuracy, model_precision, model_recall, model_f1) * 100
)

print(
  ggplot(model_heat, aes(x = Metric, y = "Diabetes Model", fill = Value)) +
    geom_tile(color = "white", linewidth = 2) +
    geom_text(aes(label = paste0(round(Value, 2), "%")),
              size = 5, fontface = "bold") +
    scale_fill_gradient(low = "lightyellow", high = "darkgreen",
                        limits = c(0, 100)) +
    labs(title = "Heatmap of Diabetes Prediction Metrics",
         x = "Evaluation Metrics", y = "", fill = "Score (%)") +
    theme_minimal()
)

## Export results

The following cell saves the model metrics and test predictions as CSV files in the Colab working directory.

In [ ]:
%%R
write.csv(results, "diabetes_prediction_results.csv", row.names = FALSE)
write.csv(
  data.frame(
    Metric = c("Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"),
    Value = c(model_accuracy, model_precision, model_recall,
              model_f1, auc_value)
  ),
  "diabetes_evaluation_metrics.csv",
  row.names = FALSE
)
cat("CSV result files saved successfully.\n")

## Important note

This is an academic demonstration of Logistic Regression for binary classification. It is not a medical diagnostic tool and must not be used for clinical decisions.